# Введение в регрессионный анализ

*Алла Тамбовцева*

## Семинар 5. Множественная линейная регрессия. Мера $R^2$

### Контекст исследования

В файле `english.csv` (скачать [здесь](https://github.com/allatambov/PyReg26/blob/main/english.csv)) сохранены результаты психолингвистического исследования, посвященного узнаваемости слов (*lexical decision & word naming*). В такого рода исследованиях участникам эксперимента предлагают определить, является ли слово, которое они видят на экране, реально существующим в языке или нет. Другими словами, настоящее это слово или что-то выдуманное, но созданное по правилам языка. Если участники узнают слово, они должны прочитать его вслух и нажать на специальную кнопку. Время, затраченное на узнавание слова фиксируется и измеряется в милисекундах: 
насколько быстро человек нажал на кнопку (реальное слово или нет, *lexical decision*), 
или прочитал слово (*word naming*).

### Переменные 

Некоторые из переменных (в файле `english.csv` отобраны только случаи с реально существующими словами, иначе грамматические характеристики ниже будут неприменимы):

* `AgeSubject`: молодой участник эксперимента или нет;
* `WordCategory`: часть речи (`N` – существительное, `V` – глагол);
* `RTlexdec`: время в милисекундах, затраченное на узнавание слова (с момента появления слова на экране до нажатия кнопки, реальное слово или нет);
* `RTnaming`: время в милисекундах, затраченное на называние слова (с момента появления слова на экране до произнесения его вслух);
* `WrittenFrequency`: мера того, насколько часто слово встречается в письменных текстах;
* `LengthInLetters`: длина слова в буквах;
* `FamilySize`: мера того, насколько богата морфологическая семья слова (как много однокоренных слов с разной частью речи);
* `NumberSimplexSynsets`: мера того, насколько у слова много синонимов.

### План нашего исследования

1. Выбрать строки, соответствующие молодым участникам эксперимента. На этой подвыборке построить модель, которая объясняет, как время, затрачиваемое на узнавание слова, зависит от его встречаемости в письменных текстах, его длины, размера морфологической семьи и количества синонимов; прокомментировать результаты и качество полученной модели.
 
2. Выбрать строки, соответствующие более старшим участникам эксперимента. На этой подвыборке построить ту же модель, что и выше. Прокомментировать полученные результаты.

### Строим модели

Импортируем необходимые библиотеки и функции:

In [1]:
import pandas as pd
from statsmodels.formula.api import ols

----

**NB.** На лекции использовался немного другой вариант импорта, будьте бдительны и внимательно читайте условия задач в проверочных. 

**Вариант 1:** импортируем весь модуль `formula.api` целиком (с сокращенным названием), а затем из него вызываем функцию `ols()`:

    import statsmodels.formula.api as smf
    smf.ols()

**Вариант 2:** импортируем из `formula.api` только одну функцию `ols()`, ее название можем писать отдельно (без префиксов):

    from statsmodels.formula.api import ols
    ols()

Внутри `ols()` код при любом импорте одинаковый, формула для модели и указание на датафрейм с данными, см. ниже.

----

Загружаем данные из файла `english.csv`:

In [2]:
df = pd.read_csv("english.csv")
df

,AgeSubject,WordCategory,RTlexdec,RTnaming,WrittenFrequency,LengthInLetters,FamilySize,NumberSimplexSynsets
0,young,N,6.543754,6.145044,3.912023,3,1.386294,0.693147
1,young,N,6.397596,6.246882,4.521789,5,1.386294,1.098612
2,young,N,6.304942,6.143756,6.505784,6,1.609438,2.484907
3,young,N,6.424221,6.131878,5.017280,4,1.945910,1.098612
4,young,N,6.450597,6.198479,4.890349,4,2.197225,2.484907
...,...,...,...,...,...,...,...,...
4563,old,V,6.753998,6.446513,2.079442,3,1.386294,1.098612
4564,old,V,6.711022,6.506979,3.663562,4,1.609438,1.386294
4565,old,V,6.592332,6.386879,5.043425,4,1.945910,2.564949
4566,old,V,6.565561,6.519884,3.135494,5,1.945910,1.609438


Убедимся, что возраст участников закодирован текстовыми значениями, и что их только два:

In [3]:
# одинаковое число в категориях young и old
# одинаковое число молодых и не очень

df["AgeSubject"].value_counts()

AgeSubject
young    2284
old      2284
Name: count, dtype: int64

Делим выборку на подвыборки – выбираем строки, соответствующие молодым и немолодым участникам эксперимента:

In [4]:
young = df[df["AgeSubject"] == "young"]
old = df[df["AgeSubject"] == "old"]

#### Молодые участники эксперимента

На основе данных `young` построим модель множественной линейной регрессии, где зависимой переменной является время, затраченное на узнавание слова (`RTlexdec`), а независимыми – встречаемость в письменных текстах (`WrittenFrequency`), его длина (`LengthInLetters`), размер морфологической семьи (`FamilySize`) и количество синонимов (`NumberSimplexSynsets`):

In [5]:
model01 = ols("RTlexdec ~ WrittenFrequency + LengthInLetters + FamilySize + NumberSimplexSynsets", 
    data = young).fit()
print(model01.summary())

                            OLS Regression Results                            
Dep. Variable:               RTlexdec   R-squared:                       0.432
Model:                            OLS   Adj. R-squared:                  0.431
Method:                 Least Squares   F-statistic:                     433.9
Date:                Mon, 09 Nov 2026   Prob (F-statistic):          3.06e-278
Time:                        19:08:26   Log-Likelihood:                 2525.2
No. Observations:                2284   AIC:                            -5040.
Df Residuals:                    2279   BIC:                            -5012.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                6.6297 

**Пояснение-напоминание** 

* Функция `ols()` создает объект типа «линейная модель», но она не выполняет оценку модели, то есть не вычисляет коэффициенты, значения t-статистик, p-value и прочее. На этом этапе функция просто резервирует место под результаты модели, создавая «контейнер».

* Метод `.fit()` оценивает модель, то есть выполняет все необходимые вычисления и сохраняет результаты внутри «контейнера». В атрибут `.params` записываются коэффициенты модели, в атрибут `.tvalues` – наблюдаемые значения t-статистики, в атрибут `.pvalues` – p-value для коэффициентов, и так далее.

* Метод `.summary()` подставляет результаты из разных атрибутов, созданных с помощью `.fit()`, в единую подробную выдачу. Вообще выдача выглядит как таблица, но раз мы ее вывели через `print()`, содержимое отображается как текст.

**Уравнение модели и интерпретация**

* Уравнение: $$\widehat{\text{RTlexdec}_i} = 6.63 - 0.03 \times \text{WrittenFrequency}_i + 0.002 \times \text{LengthInLetters}_i -0.02 \times \text{FamilySize}_i - 0.01 \times \text{NumberSimplexSynsets}_i.$$
* Согласно модели, среднее время узнавания слова, которое не встречается в письменных текстах (`WrittenFrequency` равно 0), имеет длину 0 (`LengthInLetter` равно 0), не имеет однокоренных слов и синонимов (`FamilySize` и `NumberSimplexSynsets` равны 0), составляет 6.63 мс. Звучит нереалистично, но константу из модели мы не выкидываем.
* **При прочих равных условиях,** при увеличении показателя встречаемости слова в письменных текстах на единицу время узнавания слова, в среднем, уменьшается на 0.03 мс. При прочих равных условиях в данном случае – если мы сравниваем слова одинаковой длины, с одинаковым числом однокоренных слов и синонимов, то есть слова, которые отличаются только по показателю `WrittenFrequency`.
* **При прочих равных условиях,** при увеличении длины слова на одну букву время узнавания слова, в среднем, увеличивается на 0.002 мс. Однако этот коэффициент не имеет смысла интерпретировать, p-value равно 0.245, а значит, на любом уровне значимости гипотеза о равенстве коэффициента при `LengthInLetter` нулю не отвергается. Длина слова не оказывает особого эффекта на его узнаваемость (точнее, на время, затрачиваемое на узнавание).
* **При прочих равных условиях,** при увеличении размера морфологической семьи на единицу, время узнавания слова, в среднем, уменьшается на 0.02 мс. Эффект статистически значимый, все вполне логично, чем больше однокоренных слов у слова, тем чаще оно возникает в разных контекстах, а значит, тем быстрее оно узнается.
* **При прочих равных условиях,** при увеличении количества синонимов (количества семейств синонимов) на единицу, время узнавания слова, в среднем, уменьшается на 0.01 мс. Эффект статистически значимый, и обратная связь здесь тоже ожидаема.
* $R^2$ модели равен 0.432, модель объясняет примерно 43% дисперсии зависимой переменной. То, насколько разнообразны значения времени узнавания слова в эксперименте, можно на 43% объяснить разнообразием значений перечисленных в модели факторов.

Изучим более внимательно $R^2$. По определению он вычисляется как отношение ESS (*explained sum of squares*) к TSS (*total sum of squares*):

$$R^2 = \frac{ESS}{TSS}$$

Проверим, что так и есть – запросим $R^2$, ESS и TSS из соответствующих атрибутов модели (`.rsquared`, `.ess`, `.centered_tss`) и сравним:

In [6]:
print(model01.rsquared)

ESS = model01.ess
TSS = model01.centered_tss
print(ESS / TSS)

0.43232942305788724
0.4323294230578872


> В `statsmodels` TSS называется `.centered_tss`, то есть «центрированная TSS», так как $TSS = \sum_i(y_i - \bar{y})^2$, а операция вычитания среднего и есть центрирование. Если запросить `.uncentered_tss`, получим просто $\sum_iy_i^2$, то есть сумму квадратов значений зависимой переменной.

Также мы знаем, что $TSS = ESS + RSS$, общая сумма квадратов складывается из объясненной суммы квадратов и суммы квадратов остатков (*residual sum of squares*), поэтому:

$$R^2 = \frac{ESS}{TSS} = \frac{ESS}{ESS + RSS} = 1 - \frac{RSS}{TSS}.$$

Запросим $RSS$ и проверим, что все работает как надо:

In [7]:
RSS = model01.ssr
print(1 - RSS / TSS) # то же самое

0.43232942305788724


> В `statsmodels` RSS называется `.ssr`, так сложилось из-за наличия разных аббревиатур.

На лекции также была формула, где $R^2$ выражается как отношение дисперсии предсказанной зависимой переменной к дисперсии самой зависимой переменной:

$$
R^2 = \frac{Var(\hat{y})}{Var(y)} = \frac{\sum_i(\hat{y}_i-\bar{y})^2/(n-1)}{\sum_i(y_i-\bar{y})^2/(n-1)} = \frac{\sum_i(\hat{y}_i-\bar{y})^2}{\sum_i(y_i-\bar{y})^2} = \frac{ESS}{TSS}
$$

Проверим и это:

In [8]:
var_yhat = model01.fittedvalues.var()
var_y = young["RTlexdec"].var()
print(var_yhat / var_y) # то же самое, тут приключения с представлением чисел

0.432329423057876


#### Немолодые участники эксперимента

Повторяем те же манипуляции, только теперь используем датафрейм  `old`:

In [9]:
model02 = ols("RTlexdec ~ WrittenFrequency + LengthInLetters + FamilySize + NumberSimplexSynsets", 
    data = old).fit()
print(model02.summary())

                            OLS Regression Results                            
Dep. Variable:               RTlexdec   R-squared:                       0.368
Model:                            OLS   Adj. R-squared:                  0.367
Method:                 Least Squares   F-statistic:                     332.2
Date:                Mon, 09 Nov 2026   Prob (F-statistic):          1.97e-225
Time:                        19:08:45   Log-Likelihood:                 2212.5
No. Observations:                2284   AIC:                            -4415.
Df Residuals:                    2279   BIC:                            -4386.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                6.8474 

**Уравнение модели и интерпретация**

* Уравнение: $$\widehat{\text{RTlexdec}_i} = 6.84 - 0.03 \times \text{WrittenFrequency}_i + 0.004 \times \text{LengthInLetters}_i -0.01 \times \text{FamilySize}_i - 0.02 \times \text{NumberSimplexSynsets}_i.$$
* Согласно модели, среднее время узнавания слова, которое не встречается в письменных текстах (`WrittenFrequency` равно 0), имеет длину 0 (`LengthInLetter` равно 0), не имеет однокоренных слов и синонимов (`FamilySize` и `NumberSimplexSynsets` равны 0), составляет 6.85 мс.  Все еще звучит нереалистично, но видно, что немолодые участники, в среднем, тратят больше времени на узнавание слов.
* **При прочих равных условиях,** при увеличении показателя встречаемости слова в письменных текстах на единицу время узнавания слова, в среднем, уменьшается на 0.03 мс. При прочих равных условиях в данном случае – если мы сравниваем слова одинаковой длины, с одинаковым числом однокоренных слов и синонимов, то есть слова, которые отличаются только по показателю `WrittenFrequency`.
* **При прочих равных условиях,** при увеличении длины слова на одну букву время узнавания слова, в среднем, увеличивается на 0.004 мс. Этот коэффициент имеет смысла интерпретировать, если мы примем уровень значимости равным 10% (уровень доверия 90%), так как p-value равно 0.09, а значит, на уровне значимости 10% гипотеза о равенстве коэффициента при `LengthInLetter` нулю отвергается. Коэффициент формально не 0, но, как мы видим, очень-очень маленький.
* **При прочих равных условиях,** при увеличении размера морфологической семьи на единицу, время узнавания слова, в среднем, уменьшается на 0.01 мс. Эффект статистически значимый, все вполне логично, чем больше однокоренных слов у слова, тем чаще оно возникает в разных контекстах, а значит, тем быстрее оно узнается.
* **При прочих равных условиях,** при увеличении количества синонимов (количества семейств синонимов) на единицу, время узнавания слова, в среднем, уменьшается на 0.02 мс. Эффект статистически значимый, и обратная связь здесь тоже ожидаема.
* $R^2$ модели равен 0.368, модель объясняет примерно 37% дисперсии зависимой переменной. То, насколько разнообразны значения времени узнавания слова в эксперименте, можно на 37% объяснить разнообразием значений перечисленных в модели факторов.

**Дополнение.** Если выберем уровень значимости 10%, тогда имеет смысл и доверительные интервалы для коэффициентов строить не 95%, а 90%. Для этого в `.summary()` изменим значение `alpha` (по умолчанию 0.05, отсюда `[0.025      0.975]` в выдаче):

In [10]:
print(model02.summary(alpha = 0.1))

                            OLS Regression Results                            
Dep. Variable:               RTlexdec   R-squared:                       0.368
Model:                            OLS   Adj. R-squared:                  0.367
Method:                 Least Squares   F-statistic:                     332.2
Date:                Mon, 09 Nov 2026   Prob (F-statistic):          1.97e-225
Time:                        19:08:48   Log-Likelihood:                 2212.5
No. Observations:                2284   AIC:                            -4415.
Df Residuals:                    2279   BIC:                            -4386.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|       [0.05       0.95]
----------------------------------------------------------------------------------------
Intercept                6.8474 

Так логичнее, теперь доверительные интервалы согласованы с уровнем значимости.

### Выгружаем результаты

Для получения красивых таблиц с результатами нам потребуется [библиотека](https://pypi.org/project/stargazer/) `stargazer` (появилась в Python как аналог одноименной библиотеки в R). Эта библиотека не входит в стандартный набор библиотек, устанавливаемых Anaconda, поэтому установим сами:

In [ ]:
!pip install stargazer

Обычно при установке показывается что-то подобное (проверьте, что есть подключение к Интернету, потому что пакет с библиотекой сначала должен скачаться, потом распаковаться, потом – установиться):
    
    Collecting stargazer
      Downloading stargazer-0.0.7-py3-none-any.whl.metadata (6.3 kB)
    Downloading stargazer-0.0.7-py3-none-any.whl (15 kB)
    Installing collected packages: stargazer
    Successfully installed stargazer-0.0.7

> Здесь  `pip install` – это не код Python, а внешняя команда, которая как будто бы запускается с командной строки (просто Jupyter позволяет проделать это здесь). После `pip install` мы указываем название библиотеки.

Выполним длинный импорт основной функции `Stargazer()` и подадим ей на вход список моделей, которые мы получили с помощью `statsmodels`:

In [11]:
from stargazer.stargazer import Stargazer

In [12]:
Stargazer([model01, model02])

Готово! Эта функция вернула таблицу, где указаны оценки коэффициентов модели (с отметкой значимости на разных уровнях в виде звездочек), стандартные ошибки коэффициентов (в круглых скобках) и прочие характеристики модели. По такой выдаче мы сразу сможем понять, какие коэффициенты значимо отличаются от 0 на уровне значимости 1% (`***` и p-value меньше 0.01), какие – на уровне значимости 5% (`**` и p-value меньше 0.05), какие – на уровне значимости 10% (`*` и p-value меньше 0.1).

Результат выше показался нам в виде готовой красивой выдачи за счет того, что по умолчанию `Stargazer()` возвращает код HTML для таблицы, а Jupyter умеет такой код сверстывать – преобразовывать в текст/таблицы/картинки. Чтобы выгрузить выдачу в файл для последующего редактирования или для добавления в свою курсовую работу/презентацию, нужно сохранить результат в переменную и применить метод для получения сырого кода.

**Если используете LaTeX**

In [13]:
# код ниже можно скопировать в файл .tex

res = Stargazer([model01, model02])
print(res.render_latex())

\begin{table}[!htbp] \centering
\begin{tabular}{@{\extracolsep{5pt}}lcc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{2}{c}{\textit{Dependent variable: RTlexdec}} \
\cr \cline{2-3}
\\[-1.8ex] & (1) & (2) \\
\hline \\[-1.8ex]
 FamilySize & -0.016$^{***}$ & -0.015$^{***}$ \\
& (0.003) & (0.003) \\
 Intercept & 6.630$^{***}$ & 6.847$^{***}$ \\
& (0.010) & (0.012) \\
 LengthInLetters & 0.002$^{}$ & 0.004$^{*}$ \\
& (0.002) & (0.002) \\
 NumberSimplexSynsets & -0.013$^{***}$ & -0.020$^{***}$ \\
& (0.003) & (0.004) \\
 WrittenFrequency & -0.030$^{***}$ & -0.028$^{***}$ \\
& (0.001) & (0.001) \\
\hline \\[-1.8ex]
 Observations & 2284 & 2284 \\
 $R^2$ & 0.432 & 0.368 \\
 Adjusted $R^2$ & 0.431 & 0.367 \\
 Residual Std. Error & 0.080 (df=2279) & 0.092 (df=2279) \\
 F Statistic & 433.913$^{***}$ (df=4; 2279) & 332.203$^{***}$ (df=4; 2279) \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{2}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}


**Если используете Word или его аналоги**

In [14]:
# код ниже можно скопировать в файл .tex

res = Stargazer([model01, model02])
print(res.render_html())

<table style="text-align:center"><tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr>
<tr><td style="text-align:left"></td><td colspan="2"><em>Dependent variable: RTlexdec</em></td></tr><tr><td style="text-align:left"></td><tr><td style="text-align:left"></td><td>(1)</td><td>(2)</td></tr>
<tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr>

<tr><td style="text-align:left">FamilySize</td><td>-0.016<sup>***</sup></td><td>-0.015<sup>***</sup></td></tr>
<tr><td style="text-align:left"></td><td>(0.003)</td><td>(0.003)</td></tr>
<tr><td style="text-align:left">Intercept</td><td>6.630<sup>***</sup></td><td>6.847<sup>***</sup></td></tr>
<tr><td style="text-align:left"></td><td>(0.010)</td><td>(0.012)</td></tr>
<tr><td style="text-align:left">LengthInLetters</td><td>0.002<sup></sup></td><td>0.004<sup>*</sup></td></tr>
<tr><td style="text-align:left"></td><td>(0.002)</td><td>(0.002)</td></tr>
<tr><td style="text-align:left">NumberSimplexSynsets</td><td>-0.013<s

Дальнейшие действия:

* скопировать код HTML выше в текстовый файл (можно в Jupyter создать через *File – New – Text file*)
* изменить расширение `.txt` на `.htm`, например, назвать файл `mymodels.htm`, сохранить изменения и закрыть
* найти файл в рабочей папке, кликнуть правой клавишей и открыть с помощью Word/аналогичной программы (по умолчанию файл `.htm` откроется в браузере как веб-страница, а нам нужен текст)

Путь к рабочей папке, где лежит этот файл можно узнать так:

In [15]:
import os
print(os.getcwd())

/Users/allat/Documents/Кафедра высшей математики/2026-2027/Введение в регрессионный анализ/02-practice/seminar05


Примеры разных настроек выдачи в `Stargazer` можно найти в примерах к официальной документации: https://github.com/StatsReporting/stargazer/blob/master/examples.ipynb.